In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

INPUT_PATH = '../../data/processed/4_activity_contributions.csv'
OUTPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


## Methodology: Clustering on Percentages vs Raw

> **Failed Experiment:** We initially tried clustering on *output percentages* (e.g., % Time in Combat).
>
> **Why it failed (Compositional Data Problem):**
> Percentages sum to 1.0. This creates linear dependency: if Combat goes up, Exploration *must* go down. This violates K-Means assumptions (independent Euclidean axes).
>
> **The Solution:** We cluster on **Normalized Raw Features** (the *causes*) rather than the *symptoms*.



In [3]:
# Prepare features for clustering
features = ['pct_combat', 'pct_collect', 'pct_explore']
X = df[features].fillna(0)

print(f"Features for clustering: {features}")
print(f"Shape: {X.shape}")

Features for clustering: ['pct_combat', 'pct_collect', 'pct_explore']
Shape: (3240, 3)


## Decision Record: Choosing K=3

> **Optimization Report Findings:**
>
> * **K=2:** Statistically superior (Silhouette 0.41), but practically useless. Lumped "Collectors" and "Explorers" into one "Non-Combat" blob.
> * **K=3 (Winner):** Silhouette 0.37 (Acceptable). Perfectly mapped to the 3 game design pillars (Combat, Collect, Explore).
> * **K=4+:** Resulted in over-segmentation (e.g., "Fast" vs "Slow" Explorers), too granular for the difficulty controller.



In [4]:
# K-Means clustering (K=3, PRODUCTION configuration)
print("Running K-Means (K=3)...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print("Clustering complete")
print(f"Cluster centers:\n{kmeans.cluster_centers_}")

Running K-Means (K=3)...
Clustering complete
Cluster centers:
[[0.65845085 0.04413931 0.29740984]
 [0.00998018 0.14233362 0.8476862 ]
 [0.32672967 0.18185444 0.49141589]]


In [5]:
# Map clusters to archetypes (assign each cluster to nearest archetype)
from scipy.spatial.distance import cdist

# Define ideal archetype centers
ideal_combat = np.array([1.0, 0.0, 0.0])  # 100% combat
ideal_collect = np.array([0.0, 1.0, 0.0])  # 100% collection
ideal_explore = np.array([0.0, 0.0, 1.0])  # 100% exploration
ideal_centers = np.array([ideal_combat, ideal_collect, ideal_explore])

# Calculate distances from each cluster to each ideal archetype
cluster_centers = kmeans.cluster_centers_
distances = cdist(cluster_centers, ideal_centers, metric='euclidean')

# Use Hungarian algorithm to assign clusters to archetypes (1-to-1 mapping)
from scipy.optimize import linear_sum_assignment
row_ind, col_ind = linear_sum_assignment(distances)

archetype_names = ['Combat', 'Collection', 'Exploration']
mapping = {row_ind[i]: archetype_names[col_ind[i]] for i in range(len(row_ind))}

df['archetype'] = df['cluster'].map(mapping)
print(f"\n Cluster mapping: {mapping}")
print(f"Archetype distribution:\n{df['archetype'].value_counts()}")


 Cluster mapping: {np.int64(0): 'Combat', np.int64(1): 'Exploration', np.int64(2): 'Collection'}
Archetype distribution:
archetype
Exploration    1507
Collection      989
Combat          744
Name: count, dtype: int64


### 🧮 Soft Membership Computation (PRODUCTION)
Calculate fuzzy membership to each archetype using inverse distance.


## 💡 Key Innovation: Soft Membership

> **The Problem with Hard K-Means:**
> Standard K-Means is binary. A player is either Cluster A or B. This loses nuance.
>
> **The Solution (Soft IDW):** we calculate **Inverse Distance** to *all* centers:
> `[Combat: 0.6, Collect: 0.3, Explore: 0.1]`
>
> **Benefit:** This vector becomes the input for our difficulty system, allowing for smooth, mixed-state adaptation rather than jerky mode switches.



In [6]:
# Compute soft membership via inverse distance
print("Computing soft membership...")

# 1. Calculate how far each point is from the center (Euclidean Distance)
#    Small distance = Very archetypal (e.g. Pure Combat player)
#    Large distance = Less like that archetype
distances = kmeans.transform(X)

# 2. Invert the distance so that 'Closer' becomes a 'Higher Score'
#    We add a tiny epsilon (1e-10) to prevent division by zero errors if distance is 0
inv_distances = 1 / (distances + 1e-10)

# 3. Normalize scores so they sum to 1.0 (100%)
#    This gives us a probability-like distribution (e.g. 60% Combat, 30% Collect, 10% Explore)
soft_membership = inv_distances / inv_distances.sum(axis=1, keepdims=True)

# Map to archetype names (now guaranteed to exist)
combat_idx = [k for k, v in mapping.items() if v == 'Combat'][0]
collect_idx = [k for k, v in mapping.items() if v == 'Collection'][0]
explore_idx = [k for k, v in mapping.items() if v == 'Exploration'][0]

# Assign the calculated soft scores to their respective columns
df['soft_combat'] = soft_membership[:, combat_idx]
df['soft_collect'] = soft_membership[:, collect_idx]
df['soft_explore'] = soft_membership[:, explore_idx]

print("Soft membership computed")
print(f"Mean soft_combat: {df['soft_combat'].mean():.4f}")
print(f"Mean soft_collect: {df['soft_collect'].mean():.4f}")
print(f"Mean soft_explore: {df['soft_explore'].mean():.4f}")
print(df.head())

Computing soft membership...
Soft membership computed
Mean soft_combat: 0.2701
Mean soft_collect: 0.3130
Mean soft_explore: 0.4169
                      _id_x                    userId     timestamp  \
0  6974894b48d53c4152cf142a  6974892348d53c4152cf1421  1.769265e+09   
1  6974896948d53c4152cf1461  6974892348d53c4152cf1421  1.769265e+09   
2  6974898748d53c4152cf148d  6974892348d53c4152cf1421  1.769265e+09   
3  697489a548d53c4152cf14b7  6974892348d53c4152cf1421  1.769265e+09   
4  697489c348d53c4152cf14e4  6974892348d53c4152cf1421  1.769265e+09   

              sessionId  itemsCollected  pickupAttempts  \
0  unreal_1769245003613        0.000000        0.000000   
1  unreal_1769245033617        0.230769        0.059524   
2  unreal_1769245063607        0.076923        0.023810   
3  unreal_1769245093601        0.230769        0.035714   
4  unreal_1769245123579        0.076923        0.130952   

   timeNearInteractables  enemiesHit  damageDone  timeInCombat  ...  \
0               

## 🧮 Delta Computation (PRODUCTION - NEW)
Calculate window-to-window changes in soft membership for ANFIS temporal context.


In [7]:
# Compute deltas (per player, sequential)
print("Computing temporal deltas...")

# 1. Sort the data carefully
#    We must group by User first, then sort by Time.
#    This ensures specific user's timeline is in correctness order.
df_sorted = df.sort_values(['userId', 'timestamp'])
print(df_sorted.head())

# 2. Calculate the 'Difference' (Delta) from the previous row
#    groupby('userId') ensures we don't accidentally compare User A's last move to User B's first move.
#    diff() subtracts: (Current Value - Previous Value)
#    - Positive Value: The player is doing MORE of this (Momentum is increasing)
#    - Negative Value: The player is doing LESS of this (Momentum is decreasing)
df_sorted['delta_combat'] = df_sorted.groupby('userId')['soft_combat'].diff().fillna(0)
df_sorted['delta_collect'] = df_sorted.groupby('userId')['soft_collect'].diff().fillna(0)
df_sorted['delta_explore'] = df_sorted.groupby('userId')['soft_explore'].diff().fillna(0)

df = df_sorted.copy()

print("Deltas computed (first window per player = 0)")
print(f"Δcombat range: [{df['delta_combat'].min():.3f}, {df['delta_combat'].max():.3f}]")
print(f"Δcollect range: [{df['delta_collect'].min():.3f}, {df['delta_collect'].max():.3f}]")
print(f"Δexplore range: [{df['delta_explore'].min():.3f}, {df['delta_explore'].max():.3f}]")

Computing temporal deltas...
                      _id_x                    userId     timestamp  \
0  6974894b48d53c4152cf142a  6974892348d53c4152cf1421  1.769265e+09   
1  6974896948d53c4152cf1461  6974892348d53c4152cf1421  1.769265e+09   
2  6974898748d53c4152cf148d  6974892348d53c4152cf1421  1.769265e+09   
3  697489a548d53c4152cf14b7  6974892348d53c4152cf1421  1.769265e+09   
4  697489c348d53c4152cf14e4  6974892348d53c4152cf1421  1.769265e+09   

              sessionId  itemsCollected  pickupAttempts  \
0  unreal_1769245003613        0.000000        0.000000   
1  unreal_1769245033617        0.230769        0.059524   
2  unreal_1769245063607        0.076923        0.023810   
3  unreal_1769245093601        0.230769        0.035714   
4  unreal_1769245123579        0.076923        0.130952   

   timeNearInteractables  enemiesHit  damageDone  timeInCombat  ...  \
0               0.000000    0.000000    0.000000      0.000000  ...   
1               0.155556    0.000000    0.00000

In [9]:
# ── Fix 9.4: Hard vs Soft clustering comparison ──────────────────────────────
# Validates that IDW soft membership produces smoother behavioural transitions
# than one-hot hard labels, motivating the fuzzy approach used in AURA.
#
# Metric: Mean per-user frame-to-frame transition magnitude.
#   Hard labels jump 0→1 on archetype change → high discontinuity.
#   Soft membership changes gradually → lower magnitude = smoother.
#
# A lower transition_magnitude for 'soft' confirms that soft membership
# better captures continuous behavioural drift without discrete jumps.

results = []

for tag, cols in [
    ('hard', ['hard_combat', 'hard_collect', 'hard_explore']),
    ('soft', ['soft_combat', 'soft_collect', 'soft_explore']),
]:
    temp = df.copy()

    if tag == 'hard':
        # One-hot encode the winning archetype as the hard-label baseline
        temp['hard_combat']  = (temp['archetype'] == 'Combat').astype(float)
        temp['hard_collect'] = (temp['archetype'] == 'Collection').astype(float)
        temp['hard_explore'] = (temp['archetype'] == 'Exploration').astype(float)

    temp = temp.sort_values(['userId', 'timestamp'])

    for c in cols:
        temp[f'd_{c}'] = temp.groupby('userId')[c].diff().abs().fillna(0)

    temp['transition_magnitude'] = temp[[f'd_{c}' for c in cols]].sum(axis=1)
    mean_mag = temp['transition_magnitude'].mean()

    results.append({'scheme': tag, 'mean_transition_magnitude': round(mean_mag, 4)})

result_df = pd.DataFrame(results)
print('Hard vs Soft Clustering — Transition Smoothness:')
print(result_df.to_string(index=False))

hard_mag = result_df.loc[result_df['scheme'] == 'hard', 'mean_transition_magnitude'].values[0]
soft_mag = result_df.loc[result_df['scheme'] == 'soft', 'mean_transition_magnitude'].values[0]
reduction_pct = (hard_mag - soft_mag) / hard_mag * 100
print(f'Soft membership reduces mean transition magnitude by {reduction_pct:.1f}% vs hard labels.')


Hard vs Soft Clustering — Transition Smoothness:
scheme  mean_transition_magnitude
  hard                     0.9988
  soft                     0.6559
Soft membership reduces mean transition magnitude by 34.3% vs hard labels.


In [10]:
# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n Saved to {OUTPUT_PATH}")
print("Clustering + soft membership + deltas complete!")


 Saved to ../../data/processed/5_clustered_telemetry.csv
Clustering + soft membership + deltas complete!


In [11]:
# =============================================================================
# EXPORT CLUSTER CENTROIDS TO JSON (for demo UI)
# =============================================================================
# After rerunning this notebook, copy the generated cluster_centroids.json to:
#   anfis-demo-ui/models/cluster_centroids.json
# =============================================================================

import json

CENTROIDS_OUTPUT = '../../data/processed/cluster_centroids.json'
CENTROIDS_DEPLOY = '../../../anfis-demo-ui/models/cluster_centroids.json'

# Build centroid dict in the format expected by the TypeScript engine
centroids_export = {}
for cluster_id, archetype_name in mapping.items():
    center = kmeans.cluster_centers_[cluster_id]
    centroids_export[str(cluster_id)] = {
        "archetype": archetype_name,
        "centroid": {
            "pct_combat":  float(center[0]),
            "pct_collect": float(center[1]),
            "pct_explore": float(center[2])
        }
    }

# Save to processed data folder
with open(CENTROIDS_OUTPUT, 'w') as f:
    json.dump(centroids_export, f, indent=2)

# Also write directly to the demo UI models folder
with open(CENTROIDS_DEPLOY, 'w') as f:
    json.dump(centroids_export, f, indent=2)

print("cluster_centroids.json exported:")
print(json.dumps(centroids_export, indent=2))
print(f"\nSaved to: {CENTROIDS_OUTPUT}")
print(f"Deployed to: {CENTROIDS_DEPLOY}")

cluster_centroids.json exported:
{
  "0": {
    "archetype": "Combat",
    "centroid": {
      "pct_combat": 0.6584508458557515,
      "pct_collect": 0.044139310324302586,
      "pct_explore": 0.29740984381994573
    }
  },
  "1": {
    "archetype": "Exploration",
    "centroid": {
      "pct_combat": 0.009980181516020814,
      "pct_collect": 0.14233362069180275,
      "pct_explore": 0.8476861977921765
    }
  },
  "2": {
    "archetype": "Collection",
    "centroid": {
      "pct_combat": 0.3267296749125025,
      "pct_collect": 0.1818544373374581,
      "pct_explore": 0.4914158877500393
    }
  }
}

Saved to: ../../data/processed/cluster_centroids.json
Deployed to: ../../../anfis-demo-ui/models/cluster_centroids.json
